In [13]:
del streamflow_stat_plots

In [1]:
import numpy as np
import pandas as pd
import requests
import os
import glob
import datetime as datetime
from dataIO import dataloader, webservices
# from dataIO.webservices import fetch_ca_water_data
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px
from plotly import graph_objects as go

In [2]:
import io

def fetch_ca_water_data(start_date, end_date, stations, parameters):
    base_url = 'https://wateroffice.ec.gc.ca/services/daily_data/csv/inline'
    
    # Format station and parameter lists for URL query parameters
    station_params = '&'.join([f'stations[]={station}' for station in stations])
    parameter_params = '&'.join([f'parameters[]={parameter}' for parameter in parameters])
    
    # Construct the complete URL with parameters
    url = f'{base_url}?{station_params}&{parameter_params}&start_date={start_date}&end_date={end_date}'
    
    # Send HTTP GET request to fetch data
    response = requests.get(url)
    
    # Check if request was successful
    if response.status_code == 200:
        # Read CSV data directly from response content
        data = pd.read_csv(io.StringIO(response.content.decode('utf-8')))
        return data
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None

In [3]:
headwater = pd.read_excel('headwater_sites_revised.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 


all_site_data = {}
for site in site_list:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1968-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass

https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12301250&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12302055&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12304500&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12306500&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12324400&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12324590&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12332000&startDT=1968-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=j

In [4]:
import sqlite3
conn = sqlite3.connect('BC Data/Hydat.sqlite3')
cursor = conn.cursor()
sql_statement = "SELECT * FROM sqlite_master WHERE type='table';"
cursor.execute(sql_statement)
tables = cursor.fetchall()
# for table in tables:
#     print(table[2])

canadian_stations = [
    '08LD001', '07EA007', '08NL004','09AA006','08FB006','08EC001','08EC013','08KE016','08LB020','08LB069','07FC001',
    '08FB007','08MB006','08NE039','08NB012','08LB038','07FC003','10CD004','08KD007','08EE013','08NN002','07FA005',
    '08ND012','08NB014','08NH130','08KB001','08KA007','08KA005','08KA004','08NK018','08NP001','07FB009','07EA005',
    '08EG012','08NK016','08NK002','08GA071','08LE024','08NH119','08JD006','08EE004','08EE003','08NC004','08GA072',
    '08MG001','08HA001','08MB005','08MA002','08MA001','08MH103','08MH016','08MH001','07EE009','08LA001','08LG048',
    '08NB005','08NA002','10AC005','08FC003','08KA001','08JD006','08NH119','08LE024','08GA071','08NK002','08NK016',
    '08EG012','07EA005','07FB009','08NP001','08NK018','08KA004','08KA005','08KA007','08KB001','08NH130','08NB014',
    '08ND012','07FA005','08NN002'
]
parameters = ['flow']

all_ca_site_data = fetch_ca_water_data(start_date='1968-10-01', end_date='2023-10-01', stations=canadian_stations, parameters=parameters)

query = """
    SELECT STATION_NUMBER, STATION_NAME, PROV_TERR_STATE_LOC, LATITUDE, LONGITUDE, DRAINAGE_AREA_GROSS FROM STATIONS 
    WHERE STATION_NUMBER IN ({})
""".format(','.join(['?']*len(canadian_stations)))

cursor.execute(query, canadian_stations)
station_meta_data = cursor.fetchall()
print(len(canadian_stations))
station_meta_data=pd.DataFrame(station_meta_data)
station_meta_data.columns = ['STATION_NUMBER', 'STATION_NAME', 'PROV_TERR_STATE_LOC', 'LATITUDE', 'LONGITUDE', 'DRAINAGE_AREA_GROSS']
# station_meta_data
# if all_ca_site_data is not None:
#     all_ca_site_data.head() 

80


In [5]:
mf_flows = pd.read_excel('mf_flows_appendix_a_w_loc.xlsx')
mf_flows.rename(columns={'Lat':'Long', 'Long':'Lat'}, inplace=True)
mf_flows.set_index('ID', inplace=True) # chaining .set_index() apparently doesn't work.
mf_flows.head(3)

,Name,Basin,Long,Lat,Section,H,S,A,L,ARF,D,DD,E,EE,M
ID,,,,,,,,,,,,,,,
ALF,Albeni Falls,Pend Oreille,-117.030000,48.180000,3.2,x,x,x,x,x,x,x,NaN,NaN,NaN
ANA,Anatone,Lower Snake,-116.976667,46.097222,3.5,x,x,x,x,x,NaN,NaN,NaN,NaN,NaN
BIT,Bitterroot,Pend Oreille,-114.050000,46.830000,3.2,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
modified_flows_m = {}

for file in glob.glob('modifiedflows2020/m_files/*'):
    # print(file)
    filename = file.split('/')[2]
    sitename = filename.split('_')[0]
    # print(sitename[:3])
    # print(mf_flows['ID'])
    if sitename[:3] in mf_flows.index.to_list():
        df = pd.read_excel(file)
        # print(sitename[:3])
        modified_flows_m[sitename] = df
# print(len(modified_flows_m.keys()))

In [ ]:
# a = str(pd.to_datetime('1960-10-01').date())
# b = str(pd.to_datetime('2018-10-10').date())
# f_ca[f_ca['Date'].between(a, b)]

In [33]:
# d = dataloader.DataLoader(filtered_all_ca_site_data.get('07EE009'), 'Date', 'Value/Valeur')

In [7]:
# filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date']].between(start_date, end_date)


In [6]:
end_date = '2018-10-01'
start_dates = [
#     '1938-10-01', 
#     '1948-10-01', 
    '1968-10-01',
#     '1975-10-01',
    # '1978-10-01',
    # '1988-10-01',
    # '1998-10-01', 
#     '2010-10-01', 
#     '2018-10-01', 
]
figs_hw = []
figs_ca = []
figs_mf = []
for start_date in start_dates:
# for start_date, end_date in zip(start_dates, end_dates):

###-------USGS gages/Headwater sites-----------------------------
#     file_name = f'Headwater_MK_Results_{start_date}to{end_date}.html'
    # try:
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    yrs_of_data = []
    included_sites = []
    excluded_sites = []
    for site, df in all_site_data.items():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    
        filtered_df = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Discharge']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
            excluded_sites.append(site)
        else:
            filtered_all_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
            included_sites.append(site)
    
    volume_stats = {}
    print(f'Number of included headwater sites: {len(included_sites)}')
    print(f'Number of headwater sites excluded from study: {len(excluded_sites)}')
    
    for site, site_df in filtered_all_site_data.items():
        d = dataloader.DataLoader(site_df, 'Date', name_of_Q_column='Discharge')
        s = climatestatistics.Streamflow(d)
      
        try:
            mann_kendall_results = {}
            mann_kendall_results['lat'] = headwater[headwater.index==site].iat[0,1]
            mann_kendall_results['long'] = headwater[headwater.index==site].iat[0,2]
            mann_kendall_results['name'] = headwater[headwater.index==site].iat[0,0]
            
        #     for day in ['01-01', '03-01', '05-01', '08-01']:
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            figs_hw.append(runoff_bw_dates_plot)

            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
            print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
            figs_hw.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='07-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
            figs_hw.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='08-01', end_month_day='9-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Late Summer'] = s.volume_bw_days_mann_kendall_test
            figs_hw.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Fall'] = s.volume_bw_days_mann_kendall_test
            figs_hw.append(runoff_bw_dates_plot) 
        
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='01-31', alpha=.05)            
#             mann_kendall_results['Jan']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='02-01', end_month_day='02-28', alpha=.05)
#             mann_kendall_results['Feb']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='03-01', end_month_day='03-28', alpha=.05)
#             mann_kendall_results['Mar'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='4-30', alpha=.05)
#             mann_kendall_results['Apr'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='05-01', end_month_day='05-31', alpha=.05)            
#             mann_kendall_results['May']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='06-01', end_month_day='06-30', alpha=.05)
#             mann_kendall_results['Jun']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='07-31', alpha=.05)
#             mann_kendall_results['Jul'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='08-01', end_month_day='8-31', alpha=.05)
#             mann_kendall_results['Aug'] = s.volume_bw_days_mann_kendall_test            
#             s.calc_runoff_bw_days(begin_month_day='09-01', end_month_day='09-30', alpha=.05)            
#             mann_kendall_results['Sep']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='10-31', alpha=.05)
#             mann_kendall_results['Oct']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='11-01', end_month_day='11-30', alpha=.05)
#             mann_kendall_results['Nov'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='12-01', end_month_day='12-31', alpha=.05)
#             mann_kendall_results['Dec'] = s.volume_bw_days_mann_kendall_test  

#             mann_kendall_results['Site Type'] = 'USGS'
#             volume_stats[site] = mann_kendall_results

        except Exception as e:
            raise ValueError('Saving to volume_stats_hw prob failed')
            print(e)
            

    

# ##-----------Canadian sites---------------            
    # try:
#     start_date = pd.to_datetime(start_date).date()
#     end_date = pd.to_datetime(end_date).date()
    filtered_all_ca_site_data = {}
    yrs_of_data = []
    included_sites = []
    excluded_sites = []
    for site in all_ca_site_data[' ID'].unique(): #station_meta_data['STATION_NUMBER'].unique():
        # num_yrs_of_data = len(df['Date'])/365
        # yrs_of_data.append(num_yrs_of_data)
    #fiiiiiiii
        filtered_df = all_ca_site_data[all_ca_site_data[' ID']==site]
        f_ca = filtered_df
#         pd.to_datetime(filtered_df['Date'])
        filtered_df = filtered_df[filtered_df.loc[:, 'Date'].between(str(start_date), str(end_date))]
#         print(type(filtered_df['Date']))
#         filtered_df.loc[:, 'Date'] = pd.to_datetime(filtered_df.loc[:, 'Date'])
        #if more than 15% of data is missing during the time period, exclude station
        if ((end_date - start_date).days * 0.85) > len(filtered_df['Value/Valeur']):
            print(f"Site {site} is missing more than 15% of data between {start_date} and {end_date}.  It will NOT be included in the study.")
            excluded_sites.append(site)
        else:
            filtered_all_ca_site_data[site] = filtered_df
            print(f'Site {site} has more than 85% of data between {start_date} and {end_date} and will be included in the study.')
            included_sites.append(site)
    
#     volume_stats = {}
#     print(f'Number of included headwater sites: {len(included_sites)}')
#     print(f'Number of headwater sites excluded from study: {len(excluded_sites)}')
    
    for site in all_ca_site_data[' ID'].unique(): #station_meta_data['STATION_NUMBER'].unique():
        print(site)
        try:
            d = dataloader.DataLoader(filtered_all_ca_site_data.get(site), 'Date', 'Value/Valeur')
            s = climatestatistics.Streamflow(d)
#         except:
#             pass
#         try:
            mann_kendall_results = {}
            mann_kendall_results['lat'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['LATITUDE']
            mann_kendall_results['long'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['LONGITUDE']
            mann_kendall_results['name'] = station_meta_data[station_meta_data['STATION_NUMBER']==site]['STATION_NAME']
            
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            figs_ca.append(runoff_bw_dates_plot)

            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
            print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
            figs_ca.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='07-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
            figs_ca.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='08-01', end_month_day='9-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Late Summer'] = s.volume_bw_days_mann_kendall_test
            figs_ca.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Fall'] = s.volume_bw_days_mann_kendall_test
            figs_ca.append(runoff_bw_dates_plot) 
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='01-31', alpha=.05)            
#             mann_kendall_results['Jan']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='02-01', end_month_day='02-28', alpha=.05)
#             mann_kendall_results['Feb']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='03-01', end_month_day='03-28', alpha=.05)
#             mann_kendall_results['Mar'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='4-30', alpha=.05)
#             mann_kendall_results['Apr'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='05-01', end_month_day='05-31', alpha=.05)            
#             mann_kendall_results['May']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='06-01', end_month_day='06-30', alpha=.05)
#             mann_kendall_results['Jun']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='07-31', alpha=.05)
#             mann_kendall_results['Jul'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='08-01', end_month_day='8-31', alpha=.05)
#             mann_kendall_results['Aug'] = s.volume_bw_days_mann_kendall_test            
#             s.calc_runoff_bw_days(begin_month_day='09-01', end_month_day='09-30', alpha=.05)            
#             mann_kendall_results['Sep']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='10-31', alpha=.05)
#             mann_kendall_results['Oct']=s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='11-01', end_month_day='11-30', alpha=.05)
#             mann_kendall_results['Nov'] = s.volume_bw_days_mann_kendall_test
#             s.calc_runoff_bw_days(begin_month_day='12-01', end_month_day='12-31', alpha=.05)
#             mann_kendall_results['Dec'] = s.volume_bw_days_mann_kendall_test  
            
            mann_kendall_results['Site Type'] = 'Environmental Canada'
            volume_stats[site] = mann_kendall_results
            
        except Exception as e:
#             raise ValueError('Saving to volume_stats_ca prob failed')
            print('Saving to volume_stats_ca prob failed')
            print(e)
            pass

# volume_stats_df = pd.DataFrame(volume_stats).T
            
# -------Modified Flows----------

    modified_flows_m_filtered_dates = {}
    for k,v in modified_flows_m.items():
#         print(f'dates: {start_date} and {end_date}')
#         print(f'date types: {type(start_date)} and {type(end_date)}')
#         start_date = pd.to_datetime(start_date).date()
#         end_date = pd.to_datetime(end_date).date()
#         print(f'dates: {start_date} and {end_date}')
#         print(f'date types: {type(start_date)} and {type(end_date)}')
#         print(f"type of df data: {type(modified_flows_m.get(k)['date'])}")
        
        start_date = str(start_date)
        end_date = str(end_date)
        
        # modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date']>=start_date]
        modified_flows_m_filtered_dates[k] = modified_flows_m.get(k)[modified_flows_m.get(k)['date'].between(start_date, end_date)]
    
    
    
    volume_stats_mf = {}    
    for site in modified_flows_m_filtered_dates.items():
#         print(site)
        sitename = site[0].split('_')[0]    
        # print(sitename)
        stupid = sitename.split('6')[1]
        # print(stupid)
        name_of_Q_column = f'{stupid} (unit:cfs)'
        # print(name_of_Q_column)
    #     data = webservices.usgs_streamflow().get_data(start_date='1921-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
        d = dataloader.DataLoader(site[1], 'date', name_of_Q_column=name_of_Q_column)
        s = climatestatistics.Streamflow(d)
        # print(d._df[d._name_of_Q_column][0])
        site_short = site[0][:3]
        # print(f'Site: {site_short}')
    
        try:
            mann_kendall_results_mf = {}
            
            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            figs_mf.append(runoff_bw_dates_plot)

            s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
            print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Winter Vol']=s.volume_bw_days_mann_kendall_test
            figs_mf.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='07-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test
            figs_mf.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='08-01', end_month_day='9-30', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Late Summer'] = s.volume_bw_days_mann_kendall_test
            figs_mf.append(runoff_bw_dates_plot)


            s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            # print(f'Mann-Kendall Test Results looking for trend in discharge from {s.begin_month_day} to {s.end_month_day}: {s.volume_bw_days_mann_kendall_test}')
            runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
            runoff_bw_dates_plot.plot_runoff_volume_between_2days(site)
            mann_kendall_results['Fall'] = s.volume_bw_days_mann_kendall_test
            figs_mf.append(runoff_bw_dates_plot)              
#             s.calc_max(calc_from_rolling_median=False, window_size=7)  
#             s.calc_annual_runoff_threshold_day(0.5, alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='12-31', alpha=.05)
#             s.calc_runoff_bw_days(begin_month_day='10-01', end_month_day='12-31', alpha=.05)
            
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results_mf['Fall Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='01-01', end_month_day='03-31', alpha=.05)
#             runoff_bw_dates_plot = streamflow_stat_plots.Plot_Runoff_Volume_Between_2Days(s)
#             mann_kendall_results_mf['Winter Vol']=s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='04-01', end_month_day='06-30', alpha=.05)

#             mann_kendall_results_mf['Runoff Season Vol'] = s.volume_bw_days_mann_kendall_test

#             s.calc_runoff_bw_days(begin_month_day='07-01', end_month_day='9-30', alpha=.05)

#             mann_kendall_results_mf['Summer Vol'] = s.volume_bw_days_mann_kendall_test
           
            
#             mann_kendall_results_mf['Q on date of peak Q date MK Test'] = s.rolling_yr_Qmax_mk_test
#             mann_kendall_results_mf['dayofyear on date of peak Q date MK Test'] = s.rolling_yr_DOYmax_mk_test
#             mann_kendall_results_mf['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
#             mann_kendall_results_mf['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test
#             mann_kendall_results_mf['Total Vol MK Test'] = s.total_volume_mann_kendall_test
#             # mann_kendall_results['Basin'] = basin
#             mann_kendall_results_mf['lat'] = mf_flows[mf_flows.index==site_short].iat[0,3]
#             mann_kendall_results_mf['long'] = mf_flows[mf_flows.index==site_short].iat[0,2]
#             # print(mann_kendall_results['long'])
#             mann_kendall_results_mf['name'] = mf_flows[mf_flows.index==site_short].iat[0,0]
#             volume_stats_mf[site_short] = mann_kendall_results_mf
#             mann_kendall_results_mf['Site Type'] = 'modified flows'

        except Exception as e:
            raise ValueError('Saving to volume_stats_mf prob failed')
            print(e)
 
    
    
    
    
    
    
    
#     sites = []
#     site_name = []
#     site_type = []
#     lat = []
#     long = []
    
#     months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
#     mk_period_results = {}
#     for month in months:
#         mk_period_results[f'{month}_trend'] = []
#         mk_period_results[f'{month}_h'] = []
#         mk_period_results[f'{month}_p'] = []
#         mk_period_results[f'{month}_tau'] = []
    
    
#     for site, value in volume_stats_hw.items():
        
#         sites.append(site)
#         site_name.append(value.get('name'))
#         site_type.append('headwater')
#         lat.append(value.get('lat'))
#         long.append(value.get('long'))              
                         
# #         for stat, e_list in mk_period_results:
#         for month in months:
#             mk_period_results.get(f'{month}_trend').append(value.get(month)[0])
#             mk_period_results.get(f'{month}_p').append(value.get(month)[2])
#             mk_period_results.get(f'{month}_tau').append(value.get(month)[4])


    
    
    
    
    
 
#     peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
#     peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
#     ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
#     ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
#     total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})

#     peak_runoffQ_title = 'Trend in Peak Runoff Volume: Mann-Kendall Test Results' 
#     peak_runoffDOY_title = 'Trend in Timing During Peak Runoff: Mann-Kendall Test Results'
#     ThresholdDOY_title = 'Trends in Timing of 50% Yearly Volume: Mann-Kendall Test Results'
#     ThresholdVol_title = 'Trend in Volume at 50% Yearly Flow: Mann-Kendall Test Results'
#     total_vol_title = 'Yearly Total Volume Trends: Mann-Kendall Test Results'
#     SummerVol_title = 'Trend in Late Summer Volumes (08-01 through 09-30): Mann-Kendall Test Results'
#     FallVol_title = 'Trend in Fall to early Winter Volumes (10-01 through 12-31): Mann-Kendall Test Results'
#     WinterVol_title = 'Trend in Wintertime Volumes (01-01 through 4-01): Mann-Kendall Test Results'
#     RunoffVol_title = 'Trend in Runoff Season Volumes (04-01 through 07-31): Mann-Kendall Test Results'

Site 12301250 is missing more than 15% of data between 1968-10-01 and 2018-10-01.  It will NOT be included in the study.
Site 12302055 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 12304500 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 12306500 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 12324400 is missing more than 15% of data between 1968-10-01 and 2018-10-01.  It will NOT be included in the study.
Site 12324590 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 12332000 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 12335100 is missing more than 15% of data between 1968-10-01 and 2018-10-01.  It will NOT be included in the study.
Site 12342500 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be includ

Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.8773518292022449), z=np.float64(-0.15432710172856248), Tau=np.float64(-0.01568627450980392), s=np.float64(-20.0), var_s=np.float64(15157.333333333334), slope=np.float64(-1.7709563164109075e-05), intercept=np.float64(0.07101206873934147))
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.3138606891117792), z=np.float64(-1.007154177972671), Tau=np.float64(-0.09803921568627451), s=np.float64(-125.0), var_s=15158.333333333334, slope=np.float64(-0.0007769078343241002), intercept=np.float64(0.16127301725387844))
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.9740822121735162), z=np.float64(-0.03248884445073132), Tau=np.float64(-0.00392156862745098), s=np.float64(-5.0),

Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.18907223933881823), z=np.float64(-1.313328617757116), Tau=np.float64(-0.12897959183673469), s=np.float64(-158.0), var_s=np.float64(14290.666666666666), slope=np.float64(-0.0024840042655291956), intercept=np.float64(0.5028576453686425))
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.3217296482185201), z=np.float64(-0.9909097557473052), Tau=np.float64(-0.09647058823529411), s=np.float64(-123.0), var_s=15158.333333333334, slope=np.float64(-0.0038405944971601582), intercept=np.float64(0.7285217494813456))
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.15501875531212161), z=np.float64(-1.4220258205821483), Tau=np.float64(-0.13959183673469389), s=np.float64(-171.0),

Site 08MH103 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NA002 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NB005 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NB012 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NB014 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NC004 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08ND012 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NE039 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NH119 has more than 85% of data between 1968-10-01 and 2018-10-01 and will be included in the study.
Site 08NH130 has more than 8

08KA005
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.4451725088539811), z=np.float64(0.763487844592186), Tau=np.float64(0.07450980392156863), s=np.float64(95.0), var_s=15158.333333333334, slope=np.float64(8.302724211815132e-05), intercept=np.float64(0.06135636669727578))
08KA007
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.5424012067003148), z=np.float64(0.6091859278759045), Tau=np.float64(0.0596078431372549), s=np.float64(76.0), var_s=np.float64(15157.333333333334), slope=np.float64(1.1407784135056904e-05), intercept=np.float64(0.012045383909020272))
08KB001
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.6145585959057649), z=np.float64(0.5035770889863355), Tau=np.float64(0.04941176470588235), s=np.flo

08NC004
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.5178287639890917), z=np.float64(0.6466959203388403), Tau=np.float64(0.06462585034013606), s=np.float64(76.0), var_s=np.float64(13450.0), slope=np.float64(4.05700121609214e-06), intercept=np.float64(0.0033888578661305934))
08ND012
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.1828461830167143), z=np.float64(1.3320426224799842), Tau=np.float64(0.12941176470588237), s=np.float64(165.0), var_s=15158.333333333334, slope=np.float64(3.558310376492206e-05), intercept=np.float64(0.012652203856749304))
08NE039
Mann-Kendall Test Results looking for trend in discharge from 01-01 to 03-31: Mann_Kendall_Test(trend='no trend', h=np.False_, p=np.float64(0.8327487758796983), z=np.float64(0.21117748892975358), Tau=np.float64(0.021176470588235293), s=np.float64(2

NameError: name 'modified_flows_m' is not defined

In [ ]:
file_name = 'USGS_individual_sites.html'
if os.path.exists(file_name):
    os.remove(file_name)
for fig in figs_hw:
    with open(file_name, 'a') as f:
        f.write(fig.fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

file_name = 'EnvCan_individual_sites.html'
if os.path.exists(file_name):
    os.remove(file_name)
for fig in figs_ca:
    with open(file_name, 'a') as f:
        f.write(fig.fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))

file_name = 'MF_individual_sites.html'
if os.path.exists(file_name):
    os.remove(file_name)
for fig in figs_mf:
    with open(file_name, 'a') as f:
        f.write(fig.fig.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))
        

In [28]:
volume_stats_df
trend = volume_stats_df['Jan'].apply(lambda x: x[0]).to_list()
p_val = volume_stats_df['Jan'].apply(lambda x: x[2]).to_list()
tau = volume_stats_df['Jan'].apply(lambda x: x[3]).to_list()

In [29]:
p_val

[0.3769975791490947,
 0.4929963604314964,
 0.9943609699435021,
 0.3572678429123073,
 0.33642307585176723,
 0.3087965659991987,
 0.07151057805609917,
 0.1183056066542365,
 0.26717014502974124,
 0.6459542929981457,
 0.008558886421878409,
 0.8046267471952453,
 0.32242854483688665,
 0.07151057805609917,
 0.01117287192680294,
 1.0889247175072114e-06,
 2.087904471537172e-08,
 0.003063462337335121,
 0.06306092548724385,
 0.02897192753396549,
 0.08590410553587513,
 0.09393309804459693,
 0.14646717796371345,
 0.7079733321543815,
 0.4163519362835637,
 0.03707630837435305,
 0.18162639007607662,
 0.016919535563399535,
 0.0025458545171921365,
 0.001144856164775554,
 0.21095137661039587,
 0.1983574171428546,
 0.9733082170178828,
 0.15959367256834933,
 0.22683636265300655,
 0.15959367256834933,
 0.23788953280067782,
 0.13812390972107313,
 0.11664049661843734,
 0.7936642234391675,
 0.7884645974487259,
 0.904365834981387,
 0.19560435481130378,
 0.11351728455021748,
 0.2669709555047193,
 0.0288046110966

In [7]:
volume_stats_df = pd.DataFrame(volume_stats).T

NameError: name 'volume_stats' is not defined

In [21]:
vol_stats_df_long = pd.melt(volume_stats_df, id_vars[])

SyntaxError: invalid syntax (3897298833.py, line 1)

In [10]:
vol_stats_df_long

NameError: name 'vol_stats_df_long' is not defined

In [11]:
volume_stats_df2 = volume_stats_df

In [12]:
volume_stats_df3 = volume_stats_df

In [19]:
volume_stats_df = volume_stats_df2

In [13]:
# months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
# results = {}
# volume_stats_df = volume_stats_df.reset_index()
# volume_stats_df = volume_stats_df.rename(columns={'index':'Site'})
# for month in months: 
#     volume_stats_df[[f'{month}_trend', f'{month}_h', f'{month}_p', f'{month}_z', 
#                      f'{month}_tau', f'{month}_s', f'{month}_var_s', f'{month}_slope',
#                      f'{month}_intercept']] = pd.DataFrame(volume_stats_df[month].tolist(), index=volume_stats_df.index)

# volume_stats_df = volume_stats_df.drop(columns=months)

# volume_stats_df_long = pd.melt(volume_stats_df, id_vars=['Site'],
#                               value_vars=[col for col in volume_stats_df.columns if 'trend' or 'p_value' or 'tau_value'],
#                                var_name='Month_Metric', value_name='Value')

# # volume_stats_df_long[['Month', 'Metric']] = volume_stats_df_long['Month_Metric'].str.split('_', expand=True)
# # volume_stats_df_long.drop(columns=['Month_Metric'])

In [84]:
volume_stats_df.columns

Index(['Site', 'lat', 'long', 'name', 'Site Type', 'Jan_trend', 'Jan_h',
       'Jan_p', 'Jan_z', 'Jan_tau',
       ...
       'Nov_intercept', 'Dec_trend', 'Dec_h', 'Dec_p', 'Dec_z', 'Dec_tau',
       'Dec_s', 'Dec_var_s', 'Dec_slope', 'Dec_intercept'],
      dtype='object', length=113)

In [14]:
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
results = {}
for month in months:   
    for site in volume_stats_df.index:
        results[f'{month}_trend'] = volume_stats_df[month].loc[site][0]
        results[f'{month}_p'] = volume_stats_df[month].loc[site][2]
        results[f'{month}_z'] = volume_stats_df[month].loc[site][3]
        results[f'{month}_tau'] = volume_stats_df[month].loc[site][4]
        results[f'{month}_s'] = volume_stats_df[month].loc[site][5]
        results[f'{month}_var_s'] = volume_stats_df[month].loc[site][6]
        results[f'{month}_slope'] = volume_stats_df[month].loc[site][7]
    

In [30]:
# results

In [ ]:
# peak_runoffQ = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type, 'lat': lat, 'long': long, 'trend': PeakRunoffQ_trend,  'p': PeakRunoffQ_p, 'tau': PeakRunoffQ_tau})
# peak_runoffDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': PeakRunoffDOY_trend,  'p': PeakRunoffDOY_p, 'tau': PeakRunoffDOY_tau})
# ThresholdVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdVol_trend, 'p': ThresholdVol_p, 'tau': ThresholdVol_tau})
# ThresholdDOY = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': ThresholdDOY_trend, 'p': ThresholdDOY_p, 'tau': ThresholdDOY_tau})
# total_vol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': TotalVol_trend, 'p': TotalVol_p, 'tau': TotalVol_tau})
# RunoffVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': RunoffVol_trend, 'p': RunoffVol_p, 'tau': RunoffVol_tau})
# SummerVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'site_type': site_type,'lat': lat, 'long': long, 'trend': SummerVol_trend, 'p': SummerVol_p, 'tau': SummerVol_tau})
# WinterVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type,'lat': lat, 'long': long, 'trend': WinterVol_trend, 'p': WinterVol_p, 'tau': WinterVol_tau})
# FallVol = pd.DataFrame({'sites':sites, 'site_name': site_name, 'site_type': site_type, 'lat': lat, 'long': long, 'trend': FallVol_trend, 'p': FallVol_p, 'tau': FallVol_tau})

In [25]:
# volume_stats_df#.loc[:,['Jun', 'Jul']]

In [26]:
# volume_stats_df.loc[:,'Jan'].iloc[0]

In [24]:
# meta_df = volume_stats_df.loc[:, ['lat', 'long', 'name']]
# meta_df

In [23]:
# month_stats = volume_stats_df.loc[:, month]
# month_stats.to_list()

In [22]:
# df =volume_stats_df.drop(month, axis=1)
# df

In [20]:
# volume_stats_df4 = volume_stats_df.reset_index()
# volume_stats_df4 = volume_stats_df4.rename(columns={'index':'site_id'})

In [21]:
# meta_df = volume_stats_df.loc[:, ['lat', 'long', 'name', 'Site Type']]
# meta_df

In [16]:
meta_df = volume_stats_df.loc[:, ['lat', 'long', 'name', 'Site Type']]
stats = pd.DataFrame(month_stats.to_list(), columns=['trend', 'h', 'p', 'z', 'Tau', 's', 'var_s', 'slope', 'intercept'])
#     print(stats)
stats = pd.merge(meta_df, stats, left_index=True, right_index=True)
#     stats = pd.concat([volume_stats_df.drop(month, axis=1), stats], axis=1)
#     stats = pd.concat([meta_df, stats], axis=1)

In [17]:
volume_stats_df

,site_id,lat,long,name,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Site Type
0,12302055,48.355603,-115.31465,Fisher River near Libby MT,"(no trend, False, 0.3769975791490947, -0.88344...","(no trend, False, 0.16383024552965075, -1.3923...","(no trend, False, 0.6257907526390425, -0.48765...","(decreasing, True, 0.04252122229706323, -2.028...","(decreasing, True, 0.03458425704938328, -2.113...","(decreasing, True, 0.02897192753396549, -2.183...","(decreasing, True, 0.014189316241066674, -2.45...","(no trend, False, 0.11501190973233788, -1.5760...","(decreasing, True, 0.00016522357012860311, -3....","(decreasing, True, 0.022441339818437545, -2.28...","(no trend, False, 0.893178666143057, 0.1342831...","(no trend, False, 0.5201295773063765, -0.64314...",USGS
1,12304500,48.561722,-115.970158,Yaak River near Troy MT,"(no trend, False, 0.4929963604314964, -0.68555...","(no trend, False, 0.26717014502974124, -1.1096...","(no trend, False, 0.8155838577068915, -0.23322...","(decreasing, True, 0.03581179937943069, -2.099...","(decreasing, True, 0.02328787110405961, -2.268...","(no trend, False, 0.11179077639088408, -1.5901...","(decreasing, True, 0.009890042498012619, -2.57...","(no trend, False, 0.12511453035353104, -1.5336...","(decreasing, True, 0.00020688638717691887, -3....","(decreasing, True, 0.025064410296220307, -2.24...","(no trend, False, 0.44108559892650057, -0.7703...","(no trend, False, 0.3924560862377737, -0.85517...",USGS
2,12306500,48.999167,-116.179722,MOYIE RIVER AT EASTPORT ID,"(no trend, False, 0.9943609699435021, 0.007067...","(no trend, False, 0.5574687558262528, -0.58660...","(no trend, False, 0.9718104820638074, 0.035337...","(no trend, False, 0.13222513880168085, -1.5053...","(no trend, False, 0.07151057805609917, -1.8022...","(no trend, False, 0.19104500437867866, -1.3074...","(no trend, False, 0.053676629729880077, -1.929...","(no trend, False, 0.5386369182940962, -0.61487...","(decreasing, True, 0.0005482053288650146, -3.4...","(decreasing, True, 0.04398355971361223, -2.014...","(no trend, False, 0.7937078807228737, -0.26149...","(no trend, False, 0.893178666143057, 0.1342831...",USGS
3,12324590,46.519483,-112.793172,Little Blackfoot River near Garrison MT ST,"(no trend, False, 0.3572678429123073, 0.920583...","(no trend, False, 0.5495868272974844, 0.598379...","(no trend, False, 0.48032446614667323, 0.70578...","(no trend, False, 0.932756112362773, 0.0843777...","(no trend, False, 0.7241913677645959, -0.35286...","(no trend, False, 0.878068201766014, 0.1534185...","(no trend, False, 0.5807381377006675, 0.552306...","(no trend, False, 0.6956450897588282, 0.391205...","(no trend, False, 0.12311946423625697, -1.5418...","(no trend, False, 0.3493487373444992, 0.935853...","(no trend, False, 0.0751095077140096, 1.779795...","(no trend, False, 0.05320840506825131, 1.93322...",USGS
4,12332000,46.184569,-113.501569,Middle Fork Rock Cr nr Philipsburg MT ST,"(no trend, False, 0.33642307585176723, 0.96125...","(no trend, False, 0.7665857025635621, -0.29684...","(no trend, False, 0.6922585963374686, 0.395791...","(no trend, False, 0.7291103467049747, -0.34630...","(no trend, False, 0.26112416790502824, 1.12373...","(no trend, False, 0.16383024552965075, -1.3923...","(no trend, False, 0.13589590503090587, -1.4912...","(no trend, False, 0.4666404614778705, -0.72795...","(no trend, False, 0.2551734655146509, -1.13787...","(no trend, False, 0.6257907526390425, -0.48765...","(no trend, False, 0.6510287224403535, 0.452333...","(no trend, False, 0.24069953548696743, 1.17324...",USGS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,08NK018,"53 49.893959 Name: LATITUDE, dtype: float64","53 -114.866661 Name: LONGITUDE, dtype: float64",53 FORDING RIVER AT THE MOUTH Name: STATION...,"(increasing, True, 0.008266605644713598, 2.640...","(increasing, True, 0.016971316038902007, 2.387...","(increasing, True, 0.008637922901820527, 2.626...","(no trend, False, 0.4118489667343037, -0.82064...","(no trend, Fal

In [19]:
for month in months:
    volume_stats_df = volume_stats_df3
    volume_stats_df = volume_stats_df.reset_index()
    volume_stats_df = volume_stats_df.rename(columns={'index':'site_id'})
    month_stats = volume_stats_df.loc[:, month]
    meta_df = volume_stats_df.loc[:, ['site_id', 'lat', 'long', 'name', 'Site Type']]
    del stats
    stats = pd.DataFrame(month_stats.to_list(), columns=['trend', 'h', 'p', 'z', 'Tau', 's', 'var_s', 'slope', 'intercept'])
#     print(stats)
    stats = pd.merge(meta_df, stats, left_index=True, right_index=True)
#     stats = pd.concat([volume_stats_df.drop(month, axis=1), stats], axis=1)
#     stats = pd.concat([meta_df, stats], axis=1)

#     print(stats)
    
    
    color_scale_decr = [
        [0.0, 'rgb(165,0,38)'],
        [0.05, 'rgb(215, 48, 39)'],
        [0.1, 'rgb(244,109,67)'],
        [0.15, 'rgb(253,174,97)'],
#             [.2, 'rgb(254,224,144)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

    color_scale_incr = [
        [0.0, 'rgb(0, 109, 44)'],
        [0.05, 'rgb(49, 163, 84)'],
        [0.1, 'rgb(116, 196, 118)'],
        [0.15, 'rgb(186, 228, 179)'],
#             [.2, 'rgb(237,248,233)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

    # Create bins for p values
    bins = np.linspace(0, 1, num=len(color_scale_decr))

    def get_color_decr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_decr[i][1]
        return color_scale_decr[-1][1]

    def get_color_incr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_incr[i][1]
        return color_scale_incr[-1][1]
    
    stats['trend_color1'] = stats[stats['Tau']<0]['p'].apply(get_color_decr)
    stats['trend_color2'] = stats[stats['Tau']>=0]['p'].apply(get_color_incr)
    stats['trend_color'] = stats['trend_color1'].combine_first(stats['trend_color2'])

    stats['confidence_level'] = (1 - stats['p'])*100

    stats_hw = stats[stats['Site Type']=='USGS']
    stats_hw_pos_tau = stats_hw[stats_hw["Tau"]>0]
    stats_hw_neg_tau = stats_hw[stats_hw["Tau"]<=0]

    stats_mf = stats[stats['Site Type']=='modified flows']
    stats_mf_pos_tau = stats_mf[stats_mf["Tau"]>0]
    stats_mf_neg_tau = stats_mf[stats_mf["Tau"]<=0]

    stats_ca = stats[stats['Site Type']=='Environmental Canada']
    stats_ca_pos_tau = stats_ca[stats_ca["Tau"]>0]
    stats_ca_neg_tau = stats_ca[stats_ca["Tau"]<=0]

    
    traces = []


 

In [ ]:
figs = []
small_circle = 8
large_circle = 13
#      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
for month in months:
    volume_stats_df = volume_stats_df3
    volume_stats_df = volume_stats_df.reset_index()
    volume_stats_df = volume_stats_df.rename(columns={'index':'site_id'})
    month_stats = volume_stats_df.loc[:, month]
    meta_df = volume_stats_df.loc[:, ['site_id', 'lat', 'long', 'name', 'Site Type']]
    del stats
    stats = pd.DataFrame(month_stats.to_list(), columns=['trend', 'h', 'p', 'z', 'Tau', 's', 'var_s', 'slope', 'intercept'])
#     print(stats)
    stats = pd.merge(meta_df, stats, left_index=True, right_index=True)
    
    
    color_scale_decr = [
        [0.0, 'rgb(165,0,38)'],
        [0.05, 'rgb(215, 48, 39)'],
        [0.1, 'rgb(244,109,67)'],
        [0.15, 'rgb(253,174,97)'],
#             [.2, 'rgb(254,224,144)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

    color_scale_incr = [
        [0.0, 'rgb(0, 109, 44)'],
        [0.05, 'rgb(49, 163, 84)'],
        [0.1, 'rgb(116, 196, 118)'],
        [0.15, 'rgb(186, 228, 179)'],
#             [.2, 'rgb(237,248,233)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

    # Create bins for p values
    bins = np.linspace(0, 1, num=len(color_scale_decr))

    def get_color_decr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_decr[i][1]
        return color_scale_decr[-1][1]

    def get_color_incr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_incr[i][1]
        return color_scale_incr[-1][1]
    
    stats['trend_color1'] = stats[stats['Tau']<0]['p'].apply(get_color_decr)
    stats['trend_color2'] = stats[stats['Tau']>=0]['p'].apply(get_color_incr)
    stats['trend_color'] = stats['trend_color1'].combine_first(stats['trend_color2'])

    stats['confidence_level'] = (1 - stats['p'])*100

    stats_hw = stats[stats['Site Type']=='USGS']
    stats_hw_pos_tau = stats_hw[stats_hw["Tau"]>0]
    stats_hw_neg_tau = stats_hw[stats_hw["Tau"]<=0]

    stats_mf = stats[stats['Site Type']=='modified flows']
    stats_mf_pos_tau = stats_mf[stats_mf["Tau"]>0]
    stats_mf_neg_tau = stats_mf[stats_mf["Tau"]<=0]

    stats_ca = stats[stats['Site Type']=='Environmental Canada']
    stats_ca_pos_tau = stats_ca[stats_ca["Tau"]>0]
    stats_ca_neg_tau = stats_ca[stats_ca["Tau"]<=0]
    
    traces = []


    # decreasing------------------------------------
    title=f'{month} - Trends in Volume'
    trace = go.Scattermapbox(
            mode='markers',
            lon = stats_hw_neg_tau['long'],
            lat = stats_hw_neg_tau['lat'],
    #                 name = stat['trend'],
            text = stats_hw_neg_tau['name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stats_hw_neg_tau[['site_id','p','Tau','trend','name']],
            hovertemplate =
                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
            name = 'USGS sites - neg trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stats_hw_neg_tau['p'],
                colorscale=color_scale_decr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
    #                 showscale=True,
                symbol='circle',
                )
    )

    traces.append(trace)

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_neg_tau['long'],
#             lat = stat_mf_neg_tau['lat'],
#             text = stat_mf_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stat_mf_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle'
#             )
#     )
#     traces.append(trace)  


    # increasing----------------------------------------------   

    trace = go.Scattermapbox(
            mode='markers',
            lon = stats_hw_pos_tau['long'],
            lat = stats_hw_pos_tau['lat'],
    #                 name = stat['trend'],
            text = stats_hw_pos_tau['name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stats_hw_pos_tau[['site_id','p','Tau','trend','name']],
            hovertemplate = 
               '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
            name = 'USGS sites - pos trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stats_hw_pos_tau['p'],
                colorscale=color_scale_incr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Incr Trend', y=0.75, len=0.5),
    #                 showscale=True,
                symbol='circle',
            )
    )
    traces.append(trace)


#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stats_mf_pos_tau['long'],
#             lat = stats_mf_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stats_mf_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stats_mf_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stats_mf_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#     #                 showscale=True,
#                 symbol='circle',

#         ))
#     traces.append(trace) 






# decreasing------------------------------------
#     print(stat_ca_neg_tau)
    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stats_ca_neg_tau['long']], #stat_ca_neg_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stats_ca_neg_tau['lat']], #stat_ca_neg_tau['lat'],
#                 name = stat['trend'],
        text = stats_ca_neg_tau['name'],
#             textposition='top right',
#             textfont=dict(
#                 size=20,
#                 color='black'
        customdata=stats_ca_neg_tau[['site_id','p','Tau','trend','name']],
        hovertemplate =
            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
        name = 'Canadian sites - neg trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stats_ca_neg_tau['p'],
            colorscale=color_scale_decr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
#                 showscale=True,
            symbol='circle',
            )
    )

    traces.append(trace)



# increasing----------------------------------------------   

    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stats_ca_pos_tau['long']], #stat_ca_pos_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stats_ca_pos_tau['lat']],
        text = stats_ca_pos_tau['name'],
#             textposition='top right',
#             textfont=dict(
#                 size=10,
#                 color='black'
        customdata=stats_ca_pos_tau[['site_id','p','Tau','trend','name']],
        hovertemplate = 
           '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
        name = 'Canadian sites - pos trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stats_ca_pos_tau['p'],
            colorscale=color_scale_incr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='p values'),
#                 showscale=True,
            symbol='circle',
        )
    )
    traces.append(trace)


    #--------- purely to get grey symbols on legend ----------
    #Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

#     stats_hw_nuetral = stats_hw[stats_hw["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stats_hw_nuetral['long']],
#             lat = [stats_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stats_hw_nuetral['name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stats_hw_nuetral[['site_id','p','Tau','trend','name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'USGS and EC Sites',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=small_circle,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 

#     stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_hw_nuetral['long']],
#             lat = [stat_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_hw_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 


    #------------

    layout = go.Layout(
        mapbox_layers=[
            {
                "below": 'traces',
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
                ],
            }
        ],        
        mapbox = {
    #         'style': "stamen-terrain",
            'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
            'zoom':4.5,
            'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
        },
        title=f'<b>{title}</b>',
        showlegend=True,
        legend=dict(x=0.5, y=0, traceorder='normal', orientation='h'),
        autosize=True,
        width=1100,
        height=900
    )

    fig = go.Figure(data=traces, layout=layout)
    fig.show()
    figs.append(fig)



file_name = f'mk_monthly_headwater_sites{start_date}to{end_date}.html'
if os.path.exists(file_name):
    os.remove(file_name)
for figure in figs:
    with open(file_name, 'a') as f:
        f.write(figure.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))                        

In [196]:
stats_hw_nuetral 

site_id                                               12324590
lat                                                  46.519483
long                                               -112.793172
name                Little Blackfoot River near Garrison MT ST
Site Type                                                 USGS
trend                                                 no trend
h                                                        False
p                                                     0.695645
z                                                     0.391206
Tau                                                   0.037736
s                                                         52.0
var_s                                             16995.333333
slope                                                 0.009079
intercept                                             6.290035
trend_color1                                               NaN
trend_color2                                rgb(186, 22

In [ ]:
figs = []
small_circle = 8
large_circle = 13
#      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
    if stat is peak_runoffQ:
        title=peak_runoffQ_title
    elif stat is peak_runoffDOY:
        title=peak_runoffDOY_title
    elif stat is ThresholdVol:
        title=ThresholdVol_title
    elif stat is ThresholdDOY:
        title=ThresholdDOY_title
    elif stat is total_vol:
        title=total_vol_title
    elif stat is RunoffVol:
        title=RunoffVol_title
    elif stat is SummerVol:
        title=SummerVol_title
    elif stat is WinterVol:
        title=WinterVol_title
    elif stat is FallVol:
        title=FallVol_title

#     trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}
#     stat = FallVol#, WinterVol, peak_runoffQ
#     stat['trend_numeric'] = stat['trend'].map(trend_mapping)

#         color_scale_decr = [
#             [0.0, 'rgb(128,128,128)'],
#             [0.85, 'rgb(128,128,128)'],
#             [0.851, 'rgb(254,224,144)'],
#             [0.9, 'rgb(253,174,97)'],
#             [.95, 'rgb(215, 48, 39)'],
#             [1.0, 'rgb(165,0,38)']
#         ]

#         color_scale_incr = [
#             [0.0, 'rgb(128,128,128)'],
#             [0.85, 'rgb(128,128,128)'],
#             [0.851, 'rgb(239,243,255)'],
#             [0.9, 'rgb(189,215,231)'],
#             [.95, 'rgb(49,130,189)'],
#             [1.0, 'rgb(8,81,156)']
#         ]
    color_scale_decr = [
        [0.0, 'rgb(165,0,38)'],
        [0.05, 'rgb(215, 48, 39)'],
        [0.1, 'rgb(244,109,67)'],
        [0.15, 'rgb(253,174,97)'],
#             [.2, 'rgb(254,224,144)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]

#         #blues
#         color_scale_incr = [
#             [0.0, 'rgb(8,81,156)'],
#             [0.05, 'rgb(49,130,189)'],
#             [0.1, 'rgb(107,174,214)'],
#             [0.15, 'rgb(189,215,231)'],
#             [.2, 'rgb(239,243,255)'],
#             [.21, 'rgb(128,128,128)'],
#             [1.0, 'rgb(128,128,128)']
#         ]
    #greens
    color_scale_incr = [
        [0.0, 'rgb(0, 109, 44)'],
        [0.05, 'rgb(49, 163, 84)'],
        [0.1, 'rgb(116, 196, 118)'],
        [0.15, 'rgb(186, 228, 179)'],
#             [.2, 'rgb(237,248,233)'],
        [.1501, 'rgb(128,128,128)'],
        [1.0, 'rgb(128,128,128)']
    ]


    # Create bins for p values
    bins = np.linspace(0, 1, num=len(color_scale_decr))

    def get_color_decr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_decr[i][1]
        return color_scale_decr[-1][1]

    def get_color_incr(p):
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                return color_scale_incr[i][1]
        return color_scale_incr[-1][1]

    # Apply color to each record based on p value

    stat['trend_color1'] = stat[stat['tau']<0]['p'].apply(get_color_decr)
    stat['trend_color2'] = stat[stat['tau']>=0]['p'].apply(get_color_incr)
    stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

    stat['confidence_level'] = (1 - stat['p'])*100

    stat_hw = stat[stat['site_type']=='headwater']
    stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
    stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

    stat_mf = stat[stat['site_type']=='modified flows']
    stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
    stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

    stat_ca = stat[stat['site_type']=='canadian']
    stat_ca_pos_tau = stat_ca[stat_ca["tau"]>0]
    stat_ca_neg_tau = stat_ca[stat_ca["tau"]<=0]

    traces = []


    # decreasing------------------------------------

    trace = go.Scattermapbox(
            mode='markers',
            lon = stat_hw_neg_tau['long'],
            lat = stat_hw_neg_tau['lat'],
    #                 name = stat['trend'],
            text = stat_hw_neg_tau['site_name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_neg_tau[['sites','p','tau','trend','site_name']],
            hovertemplate =
                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
            name = 'USGS sites - neg trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stat_hw_neg_tau['p'],
                colorscale=color_scale_decr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
    #                 showscale=True,
                symbol='circle',
                )
    )

    traces.append(trace)

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_neg_tau['long'],
#             lat = stat_mf_neg_tau['lat'],
#             text = stat_mf_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stat_mf_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle'
#             )
#     )
#     traces.append(trace)  


    # increasing----------------------------------------------   

    trace = go.Scattermapbox(
            mode='markers',
            lon = stat_hw_pos_tau['long'],
            lat = stat_hw_pos_tau['lat'],
    #                 name = stat['trend'],
            text = stat_hw_pos_tau['site_name'],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_pos_tau[['sites','p','tau','trend','site_name']],
            hovertemplate = 
               '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
            name = 'USGS sites - pos trending',
            showlegend=False,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color=stat_hw_pos_tau['p'],
                colorscale=color_scale_incr,
                cmin=0,
                cmax=1,
                colorbar = dict(title='P Value - Incr Trend', y=0.75, len=0.5),
    #                 showscale=True,
                symbol='circle',
            )
    )
    traces.append(trace)


#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_pos_tau['long'],
#             lat = stat_mf_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_mf_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color=stat_mf_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#     #                 showscale=True,
#                 symbol='circle',

#         ))
#     traces.append(trace) 






# decreasing------------------------------------
#     print(stat_ca_neg_tau)
    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stat_ca_neg_tau['long']], #stat_ca_neg_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stat_ca_neg_tau['lat']], #stat_ca_neg_tau['lat'],
#                 name = stat['trend'],
        text = stat_ca_neg_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=20,
#                 color='black'
        customdata=stat_ca_neg_tau[['sites','p','tau','trend','site_name']],
        hovertemplate =
            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
        name = 'Canadian sites - neg trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stat_ca_neg_tau['p'],
            colorscale=color_scale_decr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='P Value - Decr Trend', y=0.25, len=0.5),
#                 showscale=True,
            symbol='circle',
            )
    )

    traces.append(trace)



# increasing----------------------------------------------   

    trace = go.Scattermapbox(
        mode='markers',
        lon = [float(long.iloc[0]) for long in stat_ca_pos_tau['long']], #stat_ca_pos_tau['long'],
        lat = [float(lat.iloc[0]) for lat in stat_ca_pos_tau['lat']],
        text = stat_ca_pos_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=10,
#                 color='black'
        customdata=stat_ca_pos_tau[['sites','p','tau','trend','site_name']],
        hovertemplate = 
           '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
        name = 'Canadian sites - pos trending',
        showlegend=False,
        marker=go.scattermapbox.Marker(
            size=small_circle,
            color=stat_ca_pos_tau['p'],
            colorscale=color_scale_incr,
            cmin=0,
            cmax=1,
#             colorbar = dict(title='p values'),
#                 showscale=True,
            symbol='circle',
        )
    )
    traces.append(trace)


    #--------- purely to get grey symbols on legend ----------
    #Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

    stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
    trace = go.Scattermapbox(
            mode='markers',
            lon = [stat_hw_nuetral['long']],
            lat = [stat_hw_nuetral['lat']],
    #                 name = stat['trend'],
            text = [stat_hw_nuetral['site_name']],
    #             textposition='top right',
    #             textfont=dict(
    #                 size=20,
    #                 color='black'
            customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
            hovertemplate = 
               '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
            name = 'USGS and EC Sites',
            showlegend=True,
            marker=go.scattermapbox.Marker(
                size=small_circle,
                color='grey',
                colorscale=color_scale_incr,
                cmin=0,
                cmax=1,
                symbol='circle',
        ))
    traces.append(trace) 

#     stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_hw_nuetral['long']],
#             lat = [stat_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_hw_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>p value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=large_circle,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 


    #------------

    layout = go.Layout(
        mapbox_layers=[
            {
                "below": 'traces',
                "sourcetype": "raster",
                "sourceattribution": "United States Geological Survey",
                "source": [
                    "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
                ],
            }
        ],        
        mapbox = {
    #         'style': "stamen-terrain",
            'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
            'zoom':4.5,
            'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
        },
        title=f'<b>{title}</b>',
        showlegend=True,
        legend=dict(x=0.5, y=0, traceorder='normal', orientation='h'),
        autosize=True,
        width=1100,
        height=900
    )

    fig = go.Figure(data=traces, layout=layout)
    fig.show()
    figs.append(fig)



file_name = f'mk_maps_usgs_ca_sites{start_date}to{end_date}.html'
if os.path.exists(file_name):
    os.remove(file_name)
for figure in figs:
    with open(file_name, 'a') as f:
        f.write(figure.to_html(full_html=False, include_plotlyjs='cdn', default_width=1200, default_height=400))                        

In [ ]:
stat_ca

In [ ]:
traces

In [ ]:
pd.DataFrame(volume_stats_mf)

In [ ]:
# figs = []
# #      dates_ranges = {'01-01':'04-01', '04-01':'07-31', '08-01':'09-30', '10-01':'12-31'} 
# for stat in [peak_runoffQ, peak_runoffDOY, ThresholdVol, ThresholdDOY, total_vol, RunoffVol, SummerVol, FallVol, WinterVol]:
#     if stat is peak_runoffQ:
#         title=peak_runoffQ_title
#     elif stat is peak_runoffDOY:
#         title=peak_runoffDOY_title
#     elif stat is ThresholdVol:
#         title=ThresholdVol_title
#     elif stat is ThresholdDOY:
#         title=ThresholdDOY_title
#     elif stat is total_vol:
#         title=total_vol_title
#     elif stat is RunoffVol:
#         title=RunoffVol_title
#     elif stat is SummerVol:
#         title=SummerVol_title
#     elif stat is WinterVol:
#         title=WinterVol_title
#     elif stat is FallVol:
#         title=FallVol_title

# #     stat = FallVol#, WinterVol, peak_runoffQ
# #     stat['trend_numeric'] = stat['trend'].map(trend_mapping)

#     color_scale_decr = [
#         [0.0, 'rgb(165,0,38)'],
#         [0.05, 'rgb(215, 48, 39)'],
#         [0.1, 'rgb(244,109,67)'],
#         [0.15, 'rgb(253,174,97)'],
#         [.2, 'rgb(254,224,144)'],
#         [.21, 'rgb(128,128,128)'],
#         [1.0, 'rgb(128,128,128)']
#     ]

#     color_scale_incr = [
#         [0.0, 'rgb(8,81,156)'],
#         [0.05, 'rgb(49,130,189)'],
#         [0.1, 'rgb(107,174,214)'],
#         [0.15, 'rgb(189,215,231)'],
#         [.2, 'rgb(239,243,255)'],
#         [.21, 'rgb(128,128,128)'],
#         [1.0, 'rgb(128,128,128)']
#     ]


#     # Create bins for p values
#     bins = np.linspace(0, 1, num=len(color_scale_decr))

#     def get_color_decr(p):
#         for i in range(len(bins) - 1):
#             if bins[i] <= p < bins[i + 1]:
#                 return color_scale_decr[i][1]
#         return color_scale_decr[-1][1]

#     def get_color_incr(p):
#         for i in range(len(bins) - 1):
#             if bins[i] <= p < bins[i + 1]:
#                 return color_scale_incr[i][1]
#         return color_scale_incr[-1][1]

#     # Apply color to each record based on p value

#     stat['trend_color1'] = stat[stat['tau']<0]['p'].apply(get_color_decr)
#     stat['trend_color2'] = stat[stat['tau']>=0]['p'].apply(get_color_incr)
#     stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

#     stat_hw = stat[stat['site_type']=='headwater']
#     stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
#     stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

#     stat_mf = stat[stat['site_type']=='modified flows']
#     stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
#     stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

#     # trends = ['decreasing', 'no trend', 'increasing']
#     # trends_numerics = [0, 1, 2]
#     # markers = ['circle', 'circle-stroked', 'information']
#     # colors = ['red', 'grey', 'blue']
#     # site_types = ['modified flows', 'headwater']
#     # symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

#     traces = []


#     # decreasing------------------------------------

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw_neg_tau['long'],
#             lat = stat_hw_neg_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_hw_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate =
#                 '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
#             name = 'USGS sites - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=7,
#                 color=stat_hw_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values - Decr Trend', y=0.25, len=0.5),
#     #                 showscale=True,
#                 symbol='circle',
#                 )
#     )

#     traces.append(trace)

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_neg_tau['long'],
#             lat = stat_mf_neg_tau['lat'],
#             text = stat_mf_neg_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_neg_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - neg trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_mf_neg_tau['p'],
#                 colorscale=color_scale_decr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle'
#             )
#     )
#     traces.append(trace)  


#     # increasing----------------------------------------------   

#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw_pos_tau['long'],
#             lat = stat_hw_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_hw_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'USGS sites - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=6,
#                 color=stat_hw_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#     #             colorbar = dict(title='p values'),
#     #                 showscale=True,
#                 symbol='circle',
#             )
#     )
#     traces.append(trace)


#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf_pos_tau['long'],
#             lat = stat_mf_pos_tau['lat'],
#     #                 name = stat['trend'],
#             text = stat_mf_pos_tau['site_name'],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_pos_tau[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows - pos trending',
#             showlegend=False,
#             marker=go.scattermapbox.Marker(
#                 size=11,
#                 color=stat_mf_pos_tau['p'],
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values - Incr Trend', len=0.5,y=.77),
#     #                 showscale=True,
#                 symbol='circle',

#         ))
#     traces.append(trace) 


#     #--------- purely to get grey symbols on legend ----------
#     #Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

#     stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_hw_nuetral['long']],
#             lat = [stat_hw_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_hw_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_hw_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'USGS Sites',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=5,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 

#     stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = [stat_mf_nuetral['long']],
#             lat = [stat_mf_nuetral['lat']],
#     #                 name = stat['trend'],
#             text = [stat_mf_nuetral['site_name']],
#     #             textposition='top right',
#     #             textfont=dict(
#     #                 size=20,
#     #                 color='black'
#             customdata=stat_mf_nuetral[['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#             name = 'Modified Flows',
#             showlegend=True,
#             marker=go.scattermapbox.Marker(
#                 size=11,
#                 color='grey',
#                 colorscale=color_scale_incr,
#                 cmin=0,
#                 cmax=1,
#                 symbol='circle',
#         ))
#     traces.append(trace) 


#     #------------

#     layout = go.Layout(
#         mapbox_layers=[
#             {
#                 "below": 'traces',
#                 "sourcetype": "raster",
#                 "sourceattribution": "United States Geological Survey",
#                 "source": [
#                     "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#                 ],
#             }
#         ],        
#         mapbox = {
#     #         'style': "stamen-terrain",
#             'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
#             'zoom':5,
#             'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#         },
#         title=f'<b>{title}</b>',
#         showlegend=True,
#         legend=dict(x=0.5, y=-0.1, traceorder='normal', orientation='h'),
#         autosize=True,
#         width=900,
#         height=700
#     )

#     fig = go.Figure(data=traces, layout=layout)
#     fig.show()
#     figs.append(fig)

In [ ]:
# ## Works well but doesn't distinguish between increasing and decreasing trends.  Awesome scale bar for p values though.

# trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


# stat = WinterVol #peak_runoffQ
# stat['trend_numeric'] = stat['trend'].map(trend_mapping)

# # color_scale = [
# #     [0.0, 'rgb(165,0,38)'],
# #     [0.1, 'rgb(215,48,39)'],
# #     [0.2, 'rgb(244,109,67)'],
# #     [0.3, 'rgb(253,174,97)'],
# #     [0.4, 'rgb(254,224,144)'],
# #     [0.5, 'rgb(224,243,248)'],
# #     [0.6, 'rgb(171,217,233)'],
# #     [0.7, 'rgb(116,173,209)'],
# #     [0.8, 'rgb(69,117,180)'],
# #     [0.9, 'rgb(49,54,149)'],
# #     [1.0, 'rgb(2,56,88)']
# # ]
# color_scale = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# # Create bins for p values
# bins = np.linspace(0, 1, num=len(color_scale))

# def get_color(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale[i][1]
#     return color_scale[-1][1]

# # Apply color to each record based on p value
# stat['trend_color'] = stat['p'].apply(get_color)
# stat_hw = peak_runoffQ[peak_runoffQ['site_type']=='headwater']
# stat_mf = peak_runoffQ[peak_runoffQ['site_type']=='modified flows']


# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

# traces = []
# # fig = go.Figure()
# for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_hw[stat_hw["trend"]==trend]['long'],
#             lat = stat_hw[stat_hw["trend"]==trend]['lat'],
# #                 name = stat['trend'],
#             text = stat_hw[stat_hw["trend"]==trend]['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#             customdata=stat_hw[stat_hw["trend"]==trend][['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_hw[stat_hw["trend"]==trend]['p'],
#                 colorscale=color_scale,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values'),
# #                 showscale=True,
#                 symbol='circle', 
#         ))
#     traces.append(trace)


# for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
#     trace = go.Scattermapbox(
#             mode='markers',
#             lon = stat_mf[stat_mf["trend"]==trend]['long'],
#             lat = stat_mf[stat_mf["trend"]==trend]['lat'],
# #                 name = stat['trend'],
#             text = stat_mf[stat_mf["trend"]==trend]['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#             customdata=stat_mf[stat_mf["trend"]==trend][['sites','p','tau','trend','site_name']],
#             hovertemplate = 
#                '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#             marker=go.scattermapbox.Marker(
#                 size=10,
#                 color=stat_mf[stat_mf["trend"]==trend]['p'],
#                 colorscale=color_scale,
#                 cmin=0,
#                 cmax=1,
#                 colorbar = dict(title='p values'),
# #                 showscale=True,
#                 symbol='circle', 
#         ))
#     traces.append(trace)
    
    
# layout = go.Layout(
# # fig.update_layout(

# #     mapbox_layers=[
# #         {
# #             "below": 'traces',
# #             "sourcetype": "raster",
# #             "sourceattribution": "United States Geological Survey",
# #             "source": [
# #                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
# #             ],
# #         }
# #     ],        
#     mapbox = {
# #         'style': "stamen-terrain",
#         'style': 'outdoors',#'carto-positron', #'stamen-terrain',  
#         'zoom':5,
#         'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#     },
#     title=f'<b>{title}</b>',
#     showlegend=False,
#     autosize=True,
#     width=900,
#     height=700
# )

# fig = go.Figure(data=traces, layout=layout)
# fig.show()

In [ ]:
# trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


# stat = FallVol#, WinterVol, peak_runoffQ
# stat['trend_numeric'] = stat['trend'].map(trend_mapping)

# # color_scale = [
# #     [0.0, 'rgb(165,0,38)'],
# #     [0.1, 'rgb(215,48,39)'],
# #     [0.2, 'rgb(244,109,67)'],
# #     [0.3, 'rgb(253,174,97)'],
# #     [0.4, 'rgb(254,224,144)'],
# #     [0.5, 'rgb(224,243,248)'],
# #     [0.6, 'rgb(171,217,233)'],
# #     [0.7, 'rgb(116,173,209)'],
# #     [0.8, 'rgb(69,117,180)'],
# #     [0.9, 'rgb(49,54,149)'],
# #     [1.0, 'rgb(2,56,88)']
# # ]
# color_scale_decr = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# color_scale_incr = [
#     [0.0, 'rgb(8,81,156)'],
#     [0.05, 'rgb(49,130,189)'],
#     [0.1, 'rgb(107,174,214)'],
#     [0.15, 'rgb(189,215,231)'],
#     [.2, 'rgb(239,243,255)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]


# # Create bins for p values
# bins = np.linspace(0, 1, num=len(color_scale))

# def get_color_decr(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale_decr[i][1]
#     return color_scale_decr[-1][1]

# def get_color_incr(p):
#     for i in range(len(bins) - 1):
#         if bins[i] <= p < bins[i + 1]:
#             return color_scale_incr[i][1]
#     return color_scale_incr[-1][1]

# # Apply color to each record based on p value

# stat['trend_color'] = stat[stat['trend']=='decreasing']['p'].apply(get_color_decr)
# stat['trend_color'] = stat[stat['trend']=='increasing']['p'].apply(get_color_incr)

# stat_hw = stat[stat['site_type']=='headwater']
# stat_mf = stat[stat['site_type']=='modified flows']


# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

# traces = []


# # decreasing------------------------------------

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='decreasing']['long'],
#         lat = stat_hw[stat_hw["trend"]=='decreasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='decreasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='decreasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#        '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_hw[stat_hw["trend"]=='decreasing']['p'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='p values - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='decreasing']['long'],
#         lat = stat_mf[stat_mf["trend"]=='decreasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='decreasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='decreasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf[stat_mf["trend"]=='decreasing']['p'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)  


# # no trend---------------------------------------------------

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='no trend']['long'],
#         lat = stat_hw[stat_hw["trend"]=='no trend']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='no trend']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='no trend'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color='rgb(128,128,128)',
#             symbol='circle', 
#     ))
# traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='no trend']['long'],
#         lat = stat_mf[stat_mf["trend"]=='no trend']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='no trend']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='no trend'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color='rgb(128,128,128)',
#             symbol='circle', 
#     ))
# traces.append(trace)
 
    
# # increasing----------------------------------------------   
  
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw[stat_hw["trend"]=='increasing']['long'],
#         lat = stat_hw[stat_hw["trend"]=='increasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_hw[stat_hw["trend"]=='increasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw[stat_hw["trend"]=='increasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_hw[stat_hw["trend"]=='increasing']['p'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace)


# # for trend, color in zip(trends, colors):
# # for site_type, group in stat.groupby('site_type'):
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf[stat_mf["trend"]=='increasing']['long'],
#         lat = stat_mf[stat_mf["trend"]=='increasing']['lat'],
# #                 name = stat['trend'],
#         text = stat_mf[stat_mf["trend"]=='increasing']['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf[stat_mf["trend"]=='increasing'][['sites','p','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>P-value: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information

#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf[stat_mf["trend"]=='increasing']['p'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='p values - Incr Trend', len=0.5,y=.77),
# #                 showscale=True,
#             symbol='circle', 
#     ))
# traces.append(trace) 


    
    
    
    
    
    
    
    
# layout = go.Layout(
# # fig.update_layout(

#     mapbox_layers=[
#         {
#             "below": 'traces',
#             "sourcetype": "raster",
#             "sourceattribution": "United States Geological Survey",
#             "source": [
#                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#             ],
#         }
#     ],        
#     mapbox = {
# #         'style': "stamen-terrain",
#         'style': 'carto-positron',#'outdoors', #'stamen-terrain',  
#         'zoom':5,
#         'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
#     },
#     title=f'<b>{title}</b>',
#     showlegend=False,
#     autosize=True,
#     width=900,
#     height=700
# )

# fig = go.Figure(data=traces, layout=layout)
# fig.show()

In [ ]:
trend_mapping = {'decreasing': 0, 'no trend': 1, 'increasing': 2}


stat = peak_runoffQ
stat['trend_numeric'] = stat['trend'].map(trend_mapping)
stat['confidence_level'] = 1 - stat['p']

color_scale_decr = [
    [0.0, 'rgb(128,128,128)'],
    [0.85, 'rgb(128,128,128)'],
    [0.851, 'rgb(254,224,144)'],
    [0.9, 'rgb(253,174,97)'],
    [.95, 'rgb(215, 48, 39)'],
    [1.0, 'rgb(165,0,38)']
]

color_scale_incr = [
    [0.0, 'rgb(128,128,128)'],
    [0.85, 'rgb(128,128,128)'],
    [0.851, 'rgb(239,243,255)'],
    [0.9, 'rgb(189,215,231)'],
    [.95, 'rgb(49,130,189)'],
    [1.0, 'rgb(8,81,156)']
]

# color_scale_incr = [
#     [0.0, 'rgb(8,81,156)'],
#     [0.05, 'rgb(49,130,189)'],
#     [0.1, 'rgb(107,174,214)'],
#     [0.15, 'rgb(189,215,231)'],
#     [.2, 'rgb(239,243,255)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]

# color_scale_decr = [
#     [0.0, 'rgb(165,0,38)'],
#     [0.05, 'rgb(215, 48, 39)'],
#     [0.1, 'rgb(244,109,67)'],
#     [0.15, 'rgb(253,174,97)'],
#     [.2, 'rgb(254,224,144)'],
#     [.21, 'rgb(128,128,128)'],
#     [1.0, 'rgb(128,128,128)']
# ]




# Create bins for p values
bins = np.linspace(0, 1, num=len(color_scale_decr))

def get_color_decr(p):
    for i in range(len(bins) - 1):
        if bins[i] <= p < bins[i + 1]:
            return color_scale_decr[i][1]
    return color_scale_decr[-1][1]

def get_color_incr(p):
    for i in range(len(bins) - 1):
        if bins[i] <= p < bins[i + 1]:
            return color_scale_incr[i][1]
    return color_scale_incr[-1][1]

# Apply color to each record based on p value

stat['trend_color1'] = stat[stat['tau']<0]['confidence_level'].apply(get_color_decr)
stat['trend_color2'] = stat[stat['tau']>=0]['confidence_level'].apply(get_color_incr)
stat['trend_color'] = stat['trend_color1'].combine_first(stat['trend_color2'])

stat_hw = stat[stat['site_type']=='headwater']
stat_hw_pos_tau = stat_hw[stat_hw["tau"]>0]
stat_hw_neg_tau = stat_hw[stat_hw["tau"]<=0]

stat_mf = stat[stat['site_type']=='modified flows']
stat_mf_pos_tau = stat_mf[stat_mf["tau"]>0]
stat_mf_neg_tau = stat_mf[stat_mf["tau"]<=0]

# trends = ['decreasing', 'no trend', 'increasing']
# trends_numerics = [0, 1, 2]
# markers = ['circle', 'circle-stroked', 'information']
# colors = ['red', 'grey', 'blue']
# site_types = ['modified flows', 'headwater']
# symbol_map = {'modified flows': 'square', 'headwater': 'circle'}

traces = []


# decreasing------------------------------------

trace = go.Scattermapbox(
        mode='markers',
        lon = stat_hw_neg_tau['long'],
        lat = stat_hw_neg_tau['lat'],
#                 name = stat['trend'],
        text = stat_hw_neg_tau['site_name'],
#             textposition='top right',
#             textfont=dict(
#                 size=20,
#                 color='black'
        customdata=stat_hw_neg_tau[['sites','confidence_level','tau','trend','site_name']],
        hovertemplate =
            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>',     
        name = 'USGS sites - neg trending',
        showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=7,
#             color=stat_hw_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
#             ),
    #fiiiiiiiiiiii
        marker = {'size':15, 'symbol':['triangle' for i in stat_hw_neg_tau.index], 
#                   'color':stat_hw_neg_tau['confidence_level']
                 },
#         color=stat_hw_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Decr Trend', y=0.25, len=0.5),
# #                 showscale=True,
)
                           
traces.append(trace)

# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf_neg_tau['long'],
#         lat = stat_mf_neg_tau['lat'],
#         text = stat_mf_neg_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_neg_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows - neg trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=10,
#             color=stat_mf_neg_tau['confidence_level'],
#             colorscale=color_scale_decr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol='circle-stroked'
#         )
# )
# traces.append(trace)  


# # increasing----------------------------------------------   
  
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_hw_pos_tau['long'],
#         lat = stat_hw_pos_tau['lat'],
# #                 name = stat['trend'],
#         text = stat_hw_pos_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw_pos_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'USGS sites - pos trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=6,
#             color=stat_hw_pos_tau['confidence_level'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
# #             colorbar = dict(title='p values'),
# #                 showscale=True,
#             symbol=['square' for i in stat_hw_pos_tau.index],
#         )
# )
# traces.append(trace)


# trace = go.Scattermapbox(
#         mode='markers',
#         lon = stat_mf_pos_tau['long'],
#         lat = stat_mf_pos_tau['lat'],
# #                 name = stat['trend'],
#         text = stat_mf_pos_tau['site_name'],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_pos_tau[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows - pos trending',
#         showlegend=False,
#         marker=go.scattermapbox.Marker(
#             size=11,
#             color=stat_mf_pos_tau['confidence_level'],
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             colorbar = dict(title='Confidence - Incr Trend', len=0.5,y=.77),
# #                 showscale=True,
#             symbol=['square' for i in stat_mf_pos_tau.index],

#     ))
# traces.append(trace) 


#--------- purely to get grey symbols on legend ----------
#Just ploting one single point that is without a trend (p>0.6) and then using these traces for the legend.

# stat_hw_nuetral = stat_hw[stat_hw["p"]>.6].iloc[0,:]
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = [stat_hw_nuetral['long']],
#         lat = [stat_hw_nuetral['lat']],
# #                 name = stat['trend'],
#         text = [stat_hw_nuetral['site_name']],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_hw_nuetral[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'USGS Sites',
#         showlegend=True,
#         marker=go.scattermapbox.Marker(
#             size=5,
#             color='grey',
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             symbol='circle',
#     ))
# traces.append(trace) 

# stat_mf_nuetral = stat_mf[stat_mf["p"]>.6].iloc[0,:]
# trace = go.Scattermapbox(
#         mode='markers',
#         lon = [stat_mf_nuetral['long']],
#         lat = [stat_mf_nuetral['lat']],
# #                 name = stat['trend'],
#         text = [stat_mf_nuetral['site_name']],
# #             textposition='top right',
# #             textfont=dict(
# #                 size=20,
# #                 color='black'
#         customdata=stat_mf_nuetral[['sites','confidence_level','tau','trend','site_name']],
#         hovertemplate = 
#            '<b>%{text}</b><br>ID : %{customdata[0]}<br>Trend: %{customdata[3]}<br>Confidence Level: %{customdata[1]:.2f}<br>tau: %{customdata[2]:.2f}<br><extra></extra>', # extra removes trace information
#         name = 'Modified Flows',
#         showlegend=True,
#         marker=go.scattermapbox.Marker(
#             size=11,
#             color='grey',
#             colorscale=color_scale_incr,
#             cmin=0,
#             cmax=1,
#             symbol='circle',
#     ))
# traces.append(trace) 


#------------
  
layout = go.Layout(
#     mapbox_layers=[
#         {
#             "below": 'traces',
#             "sourcetype": "raster",
#             "sourceattribution": "United States Geological Survey",
#             "source": [
#                 "https://basemap.nationalmap.gov/arcgis/rest/services/USGSHydroCached/MapServer/tile/{z}/{y}/{x}"
#             ],
#         }
#     ],        
    mapbox = {
#         'style': "stamen-terrain",
        'style':'outdoors',
#         'style': 'carto-positron',  
        'zoom':4,
        'center': go.layout.mapbox.Center(lat=46.1, lon=-117.5)
    },
    title=f'<b>{title}</b>',
    showlegend=True,
    legend=dict(x=0.5, y=0, traceorder='normal', orientation = 'h'),
    autosize=True,
    width=900,
    height=700
)

fig = go.Figure(data=traces, layout=layout)
fig.show()

In [ ]:
stat_hw_nuetral

In [ ]:
stat

In [ ]:
stat_hw[stat_hw['sites']==13338500]

In [ ]:
stat_hw[stat_hw["p"]>.6].iloc[0,:]